## **LangChain**

 <a href="#LangChain-concepts">LangChain concepts</a>
        <ol>
            <li><a href="#Model">Model</a></li>
            <li><a href="#Chat-model">Chat model</a></li>
            <li><a href="#Chat-message">Chat message</a></li>
            <li><a href="#Prompt-templates">Prompt templates</a></li>
            <li><a href="#Example-selectors">Example selectors</a></li>
            <li><a href="#Output-parsers">Output parsers</a></li>
            <li><a href="#Documents">Documents</a></li>
            <li><a href="#Memory">Memory</a></li>
            <li><a href="#Chains">Chains</a></li>
            <li><a href="#Agents">Agents</a></li>
        </ol>

In [39]:
import dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import  PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.prompts import FewShotPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


### 1 & 2. Model and Chat Model 

In [2]:
Model_API = dotenv.get_key("../.env", "GEMINI_API")

In [3]:
def get_chat_llm (params=None):
    
    llm = ChatGoogleGenerativeAI(
        model = "gemini-3-flash-preview",
        temperature=0.3,
        api_key = Model_API
    )
    return llm

In [4]:
llm_model = get_chat_llm()

### 3. Chat Message 

In [5]:
message = llm_model.invoke([
    SystemMessage("You are a helpful assistant to find best car based on the user requirement in a one short sentence"),
    HumanMessage("I am more interested about sport sedan. please suggest me a one")
])

In [6]:
print(message)

content=[{'type': 'text', 'text': 'The BMW M3 is the ultimate benchmark for performance, handling, and everyday usability in a sport sedan.', 'extras': {'signature': 'ErUJCrIJAQw51sfG1ja+ofhvkBGG2C+Ioo1bAbbr9sIx6gLN1lS2Op8jnZrGW7oShdCPwu3RCj5aJxx3uGZpmIMK8A+kptEllTlCVQjDz5HlPj6KXQ3F1vtebrp5f4CMfrfpOJNl9b1zCuvlvuMZ5DSxGKINiEhWKAYjNCgaKVGv/T3jVEKMIL0rXXRhP0F1Fx/eJBhMc7+r5MgyHxQanc+HsLTWkfOKQDFH+91HuEMdR1MTDQfSq2/cCcVSeF/DRbRK06C3xG2GpSvPtZNte40EtYDaAunYMzZanKP+TQdaJL6DvfQfcGxYdUOtuovJSXg7zaaDptC80ztoUttbEsklw9zaJQMLFUUQtXLazd++EmnnL+TDpYclxV5wknQVyKovSOXCXxPwN2q8AjXJWtUWpFtyywbVAsH2ZLgx6REokTXNr9eWI2BIotFdRDL/Hz5sutmWpzbbtK1JQZA7KfoYFQcF4Vk/i/cxqizZEgQzBdTVg3pGGw73da3rPd6EmwIludH5PR/4C18eUau0jSHFGIbpIzMfQ6w3t8gUi4qOPNJwxNIu2YWet4uhHkU9q2IiNk6VobbQTMGEmShWpQ12PuNpwuPAeFnKOw5NC+vb75conLy6ACiGSZWsQyAf6ecwzxGLKOltFg+vcMjnYfWZ98UKQ2vfKlmKfeUpQKaacNUy/jiWn1GLZ+4+KTLKb2weIiVcTrJFGj1cLQMQgJ5I1VEUG72VWF8wDzJLxfuFATYjLjEmJqiucPrwegiVrdDzQPfpTaWsIEKGB8xj6SrtnkEvyzBcYFS4i2yQpRyQuD94hcZSUM27O5e0zYS2Q

### 4. Prompt templates

We use prompt templates to give single or multi messages to the model to get more clear and accurate reponse for our questions.

##### Promt Template

**Prompt Template** uses to give single string input for the model as a message 

In [7]:
prompt = PromptTemplate.from_template("Tell me a {type} about the {topic}")
input_ = {"type":"joke", "topic":"computer"}

prompt.invoke(input_)

StringPromptValue(text='Tell me a joke about the computer')

In [8]:
output = prompt | llm_model
output

PromptTemplate(input_variables=['topic', 'type'], input_types={}, partial_variables={}, template='Tell me a {type} about the {topic}')
| ChatGoogleGenerativeAI(profile={'name': 'Gemini 3 Flash Preview', 'release_date': '2025-12-17', 'last_updated': '2025-12-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-3-flash-preview', temperature=0.3, client=<google.genai.client.Client object at 0x000002BE35AE3410>, default_metadata=(), model_kwargs={})

##### Chat Prompt Template

These prompt templates are used to format a list of messages. These "templates" consist of a list of templates themselves.


In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "tell me a joke about {topic}")
])

input_ = {"topic":"cat"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='tell me a joke about cat', additional_kwargs={}, response_metadata={})])

#### Message PlaceHolder

Message Placeholder, we use to apply relevant list of messages to a paticular positions in the chat templates.

In [10]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant to user"),
    MessagesPlaceholder("msgs")
])

input_ = {
    "msgs": [
        HumanMessage(content='What is the world largest mountain?'),
        HumanMessage(content="Please give me a short description about toyota supra")
    ]
}

formatted_prompt = chat_prompt.invoke(input_)
formatted_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant to user', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the world largest mountain?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Please give me a short description about toyota supra', additional_kwargs={}, response_metadata={})])

In [11]:
response = llm_model.invoke(formatted_prompt)

In [12]:
print(response.content[0]['text'])

### The World’s Largest Mountain
The answer depends on how you measure "largest":

*   **Highest Altitude (Above Sea Level):** **Mount Everest** is the most famous, with its peak reaching **29,032 feet (8,849 meters)** above sea level.
*   **Tallest (Base to Peak):** **Mauna Kea** in Hawaii is technically taller. While its peak is only 13,803 feet above sea level, its base sits on the ocean floor. From the very bottom to the top, it measures over **33,500 feet (10,210 meters)**.
*   **Largest by Mass/Volume:** **Mauna Loa** (also in Hawaii) is the world’s largest volcano and mountain in terms of sheer bulk, covering a massive area of the ocean floor.

***

### Toyota Supra: A Short Description
The **Toyota Supra** is an iconic Japanese sports car and grand tourer that first debuted in 1978. It is best known for two distinct eras:

1.  **The Legend (Fourth Generation/A80):** Produced in the 1990s, this model became a pop-culture icon (notably in *The Fast and the Furious*). It is world-

In [13]:
chain = chat_prompt | llm_model
response = chain.invoke(input_)
print(response.content[0]['text'])

Here are the answers to your questions:

### The World’s Largest Mountain
The answer depends on how you measure "largest":

*   **Highest Altitude:** **Mount Everest** is the highest mountain above sea level, reaching **29,032 feet (8,849 meters)**.
*   **Tallest (Base to Peak):** **Mauna Kea** in Hawaii is technically the tallest. While its peak is only 13,803 feet above sea level, it extends deep onto the ocean floor. From its underwater base to its summit, it measures over **33,500 feet (10,210 meters)**.
*   **Largest by Mass:** **Mauna Loa** (also in Hawaii) is the largest mountain in terms of volume and area covered, as it is a massive shield volcano.

---

### Toyota Supra: A Short Description
The **Toyota Supra** is an iconic high-performance sports car and grand tourer produced by Toyota. 

*   **Legendary Status:** It gained worldwide fame in the 1990s with the **fourth-generation (A80)** model. This version became a pop-culture icon through movies like *The Fast and the Furi

### 5. Example Selector

If you want to provide your model with examples and you have a large number of them, you need to filter out and select the most relevant examples for a given query. This task is performed by **Example Selectors**.

There are main examples selectors:

* `Similarity`: Uses semantic similarity between inputs and examples to decide which examples to choose.
* `MMR`: Uses Max Marginal Relevance between inputs and examples to decide which examples to choose.
* `Length`: Selects examples based on how many can fit within a certain length
* `Ngram`: Uses ngram overlap between inputs and examples to decide which examples to choo

In [14]:
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input":"windy", "output":"calm"}
]
example_prompt = PromptTemplate(
    template= "input: {input}\noutput: {output}",
    input_variables=['input', 'output']
)
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=25
)

## for example driven template, we use fewshot prompt template
dynamic_prompt = FewShotPromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
    prefix = "Give the antonym of every input",
    suffix = "input: {input}\noutput:",
    input_variables = ['input']
)

In [15]:
print(dynamic_prompt.format(input = "big"))

Give the antonym of every input

input: happy
output: sad

input: tall
output: short

input: energetic
output: lethargic

input: sunny
output: gloomy

input: windy
output: calm

input: big
output:


In [16]:
long_string = "big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else"
print(dynamic_prompt.format(input = long_string))

Give the antonym of every input

input: happy
output: sad

input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else
output:


### 6. Output Parsers

Output parsers are used to convert the raw responses generated by an LLM into a more structured and useful format. They are especially helpful when working with structured data or when standardizing outputs from different language and chat models.

LangChain provides a variety of output parsers to handle different formatting needs. In this lab, you will work with the following two parsers:

* JSON Parser: Produces output in JSON format according to a specified schema. It can also utilize a Pydantic model to generate structured JSON data, making it one of the most dependable options for obtaining structured output without relying on function calling.

* CSV Parser: Formats the model's output as a list of comma-separated values (CSV), which is useful for tabular data representation.

In [17]:
class Car(BaseModel):
    query:str = Field(description="question to setup a car")
    response:str = Field(description="answer to resolve the car")

In [18]:
car_query = "tell me the information about Lambogini"

output_parser = JsonOutputParser(pydantic_object=Car)
format_instruction = output_parser.get_format_instructions()

prompt  = PromptTemplate(
    template= "Answer the user query.\n {format_instruction}\n{query}",
    input_variables=['query'],
    partial_variables= {"format_instruction": format_instruction}
)

chain = prompt | llm_model | output_parser

response = chain.invoke({"query":car_query })
print(response)


{'query': 'tell me the information about Lambogini', 'response': "Lamborghini is an Italian manufacturer of luxury sports cars and SUVs based in Sant'Agata Bolognese. Founded in 1963 by Ferruccio Lamborghini to compete with Ferrari, the company is now a subsidiary of the Volkswagen Group through Audi. Lamborghini is world-renowned for its high-performance vehicles, iconic wedge-shaped designs, and powerful engines, with famous models including the Miura, Countach, Diablo, Murciélago, Aventador, Huracán, and the Urus SUV."}


In [19]:
user_query = "Give me details about DPO fine-tune"

output_parser = CommaSeparatedListOutputParser()
format_instruction = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="Answer the user query\n{format_instruction}\n{query}",
    input_variables=["query"],
    partial_variables={"format_instruction":format_instruction}
)

chain = prompt | llm_model | output_parser

response = chain.invoke({"query":user_query})
print(response)

['Direct Preference Optimization', 'RLHF alternative', 'preference pairs', 'chosen vs rejected', 'no reward model', 'stable training', 'computationally efficient', 'KL divergence', 'log-likelihood', 'reference model', 'policy optimization', 'Bradley-Terry model', 'alignment', 'fine-tuning', 'binary cross-entropy loss', 'implicit reward function', 'supervised learning approach', 'human feedback', 'model alignment', 'iterative optimization', 'parameter efficiency', 'language model tuning', 'preference data', 'ranking loss', 'probability ratio.']


## **LangChain Components for RAG Application**

### 7. Documents

#### Document Object

The document object contains two main components:

1. page-content: this represents the content of the whole document.
2. metadata: this represents the all the attributes that paticularlly unique for the each document. this availables at Dict types

examples:
document_id, timestamp, document_title, author, etc

In [20]:
document = Document(
    page_content="This is a example for document attribute which is page_content",
    metadata={
        "document_id": 1294857,
        "document_source": "about firewall",
        "created_time": 21344656
    }
)

document.page_content

'This is a example for document attribute which is page_content'

#### Document Loaders

Document loaders in LangChain are used to import data from multiple sources. For example, you can load a PDF file and make it readable for an LLM using LangChain.

LangChain provides more than 100 different document loaders, along with integrations with major platforms like AirByte and Unstructured. These integrations allow you to load various types of content such as HTML, PDFs, and source code from different locations, including private S3 buckets and public websites.

A full list of supported document types can be found here: https://python.langchain.com/v0.1/docs/integrations/document_loaders/

In [35]:
file_path = "https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf"
loader = PyPDFLoader(file_path)

In [36]:
loader

In [38]:
document = loader.load()
document[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-06T10:08:55+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-06T10:08:55+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. It provides tool

#### URL and Website Loaders

In [40]:
web_loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
web_data = web_loader.load()

In [43]:
web_data[0].page_content[:1000]

'LangChain overview - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageLangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.Copy pageDocumentation IndexFetch the comp